# Is the leak-free network running DD?

**Jobs `50075339` (round-0 interventions) and `50077245` (chaining interventions), both three-armed.**

Every previous answer to this question in this project came from a decoding score. Finding 8's `next_node` probe correlated at Spearman +0.96 with solve_rate and was then retired outright when a hand rule that does no deduction at all (`min_unknown`, 0.632) beat the model's own features (0.545). So this run uses interventions instead.

## The eval

Take a node `v` that DD forces in some round: its known neighbours already use `q-1 = 3` of the 4 colours, and **exactly one** of them, `w`, holds the colour `a` that is doing the work, so `v` must take the one remaining colour `c`. Recolour `w` to `c` — **a single token**, asserted by `build_pair` as `(clean != corrupt).sum() == 1`. Now `a` is free and DD's answer for `v` changes from `c` to `a`. Does the model's prediction follow?

- **Round 0** (`sudoku_circuits.py`): `w` is a **hint**, given in the prompt. Tests the elimination rule over *given* facts.
- **Chained** (`sudoku_chain.py`): `w` was **deduced by the model** in an earlier round, so the edited token sits in the reasoning section. Tests whether it eliminates against facts *it produced* — propagation, not a single rule application.

Chance is `1/n_colors = 0.25`. This cannot be passed by matching the prompt: the two prompts differ in one token and the correct answers differ.

## Headline

**YES for DD's inference, NO for DD's search.** The leak-free model applies the elimination rule on **350/350** round-0 counterfactuals and chains it off its own deductions at **0.980** with no decay from round 1 to round 4. But it emits events in ascending node index for 100% of rounds, so it is not running DD's frontier schedule.

In [ ]:
import json, os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SCRATCH = os.environ["SCRATCH"]
CIRC = f"{SCRATCH}/spin_gnn/sudoku_circuits_leakfree_2026-10-28"
CHAIN = f"{SCRATCH}/spin_gnn/sudoku_chain_2026-10-28"
ARMS = [("leak-free 1.0000", "A_leakfree_1000"),
        ("leak-free 0.8540", "B_leakfree_0854"),
        ("leaked 0.9960", "C_leaked_0996")]
CHANCE = 0.25
SITES = ["edited_hint", "graph_row_u", "graph_row_w", "predict_pos"]

beh = {k: pd.read_csv(f"{CIRC}/{d}/behavior.csv") for k, d in ARMS}
patch = {k: pd.read_csv(f"{CIRC}/{d}/patching.csv") for k, d in ARMS}
heads = {k: pd.read_csv(f"{CIRC}/{d}/head_ablation.csv") for k, d in ARMS}
chain = {}
for k, d in ARMS:
    with open(f"{CHAIN}/{d}/chain_summary.json") as f:
        chain[k] = json.load(f)
print({k: len(v) for k, v in beh.items()})

## Metrics table

`clean` is accuracy on the unmodified prompt. A high `clean` with a low `flipped` means the model is right for a reason other than the elimination — which is exactly the leaked arm's profile. `reverts_to_original` is the "detected the edited prefix" signature and only applies to the chained arm.

In [ ]:
rows = []
for k, _ in ARMS:
    b, c = beh[k], chain[k]
    rows.append({"arm": k, "test": "round 0 (hint)", "n": len(b),
                 "clean": b.clean_correct.mean(), "flipped": b.corrupt_flipped.mean(),
                 "reverts": np.nan,
                 "logit_clean": b.clean_logit_diff.mean(),
                 "logit_corrupt": b.corrupt_logit_diff.mean()})
    rows.append({"arm": k, "test": "chained (deduced)", "n": c["n_cases"],
                 "clean": c["clean_correct"], "flipped": c["corrupt_flipped"],
                 "reverts": c["reverts_to_original"],
                 "logit_clean": np.nan, "logit_corrupt": np.nan})
metrics = pd.DataFrame(rows)
metrics["verdict"] = np.where(metrics.flipped >= 0.60, "uses/chains the rule", "near chance")
display(metrics.round(3))

## Key plot 1 — the rule holds on given facts *and* on facts it derived itself

The right panel is the chaining claim. A model that chained one step but not three would show round 1 high and the later rounds falling off. It does not happen; if anything the rate rises with depth. The deep bins are small because deeply chained forcings are rare, not because anything was filtered.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(ARMS))
r0 = [beh[k].corrupt_flipped.mean() for k, _ in ARMS]
ch = [chain[k]["corrupt_flipped"] for k, _ in ARMS]
cols = ["#1f77b4", "#2ca02c", "#d62728"]
ax1.bar(x - 0.2, r0, 0.38, color=cols, edgecolor="k", label="round 0 (hint)")
ax1.bar(x + 0.2, ch, 0.38, color=cols, edgecolor="k", hatch="///", alpha=0.75,
        label="chained (deduced)")
ax1.axhline(CHANCE, color="k", ls="--")
ax1.set_xticks(x); ax1.set_xticklabels([k for k, _ in ARMS], fontsize=9)
ax1.set_ylabel("flipped to DD's new answer"); ax1.set_ylim(0, 1.1)
ax1.legend(fontsize=9)

br = chain["leak-free 1.0000"]["by_round"]
ks = sorted(int(k) for k in br)
ax2.plot(ks, [br[str(k)]["flipped"] for k in ks], "o-", lw=3, ms=11, color="#1f77b4")
ax2.axhline(CHANCE, color="k", ls="--")
ax2.set_xlabel("DD round the node was deduced in"); ax2.set_xticks(ks)
ax2.set_ylabel("flipped"); ax2.set_ylim(0, 1.1)
ax2.set_title("no decay with depth (leak-free 1.0000)")
for k in ks:
    ax2.annotate(f"n={br[str(k)]['n']}", (k, br[str(k)]["flipped"]),
                 textcoords="offset points", xytext=(0, -26), ha="center", fontsize=9)
plt.tight_layout(); plt.show()

## Key plot 2 — where the recoloured fact travels

`recovery` is how much of the flip is restored by patching the clean activation at that (layer, site). **Pre-registration (c) was falsified**: `graph_row_u` and `graph_row_w` recover 0.000 at every layer in all three arms, so the deduction never routes through the adjacency table rows. L11 at the prediction position is the built-in positive control and must read 1.00.

In [ ]:
for k, _ in ARMS:
    m = patch[k].groupby(["layer", "site"]).recovery.mean().unstack()[SITES]
    print(f"\n=== {k} ===")
    display(m.round(3))

## Key plot 3 — no single head carries it

Each head is zeroed one at a time by writing its `out_proj` weight columns. `nn.MultiheadAttention` takes the functional fast path, so module hooks on `out_proj` never fire — an earlier version of this analysis silently reported `effect_retained == 1.0` for all 7200 rows. Removing the single worst head still leaves ~84% of the effect.

In [ ]:
for k, _ in ARMS:
    h = heads[k].groupby(["layer", "head"]).effect_retained.mean().sort_values()
    top = h.head(5)
    print(f"{k:20s} worst 5: " + "  ".join(
        f"L{int(l):02d}H{int(hd)}={v:.3f}" for (l, hd), v in top.items()))

## Comparison to the previous best answer on this question

There is no previous *causal* measurement of the leak-free task — it had never been run, which is why this job exists. The comparison that does exist is **finding 2** of this project ("the model is not running DD"), which rested on probes, an algorithm-ID test, a counterfactual and an SAE, all measured on the **leaked** 0.9960 checkpoint. That counterfactual is reproduced here as arm C.

| | finding 2 (leaked 0.9960) | this run (leak-free 1.0000) |
|---|---|---|
| round-0 flip rate | 0.388 (≈ chance 0.25) | **1.000** |
| chained flip rate | 0.008 | **0.980** |
| reverts to original, chained | 0.992 | 0.020 |

**Finding 2 stands for the leaked model and is overturned for the leak-free one.** The leaked model's chained arm is the sharpest result in the set: shown an edited deduction it keeps its own answer 99.2% of the time, which is precisely what a model reading finding 16's adjacency-order leak would do, since the graph row still encodes the original colouring.

**Pre-registration (d) was falsified informatively.** I predicted A > B > C monotone in solve_rate. It is A = B ≫ C: the 0.8540 leak-free model applies *and* chains the rule exactly as well as the 1.0000 one, so the last 0.146 of solve_rate is bought by something other than the forcing rule. The real boundary is leak-free vs leaked.

## What it does *not* do

The search half of DD is absent. Events are emitted in **ascending node index for 100% of rounds** (`scripts/order_baseline_next_node.py`), and on the same max-over-v F1, a hand rule that does no deduction at all — `min_unknown`, "smallest index above the previous whose colour is unknown" — scores **0.632** against the model's own `next_node` features at **0.545**. Finding 24 adds the training-side control: permuting event order within rounds costs nothing resolvable (−0.0147, p=0.225).

So the working picture is: **sweep nodes in index order; at each, eliminate the colours of its known neighbours, including ones it just derived.** DD's propagation, not DD's schedule.

## Provenance

In [ ]:
import subprocess
print("git commit :", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                     capture_output=True, text=True).stdout.strip())
print("jobs       : 50075339 (round 0, COMPLETED 00:05:53), 50077245 (chained, COMPLETED 00:01:52)")
print("sbatch     : scripts/slurm/sudoku2_circuits_leakfree.sbatch, scripts/slurm/sudoku2_chain.sbatch")
for k, _ in ARMS:
    c = chain[k]
    print(f"{k:20s} seed_start={c['seed_start']}  checkpoint={os.path.basename(c['checkpoint'])}")
print("\nconfig: N=24, degree=5, n_colors=4, rho=0.60/rho_max=0.95, d_model=256, L=12, ff=1024,")
print("        shuffle_adj=True for both leak-free arms (read from each checkpoint's own config).")
print("deck  : notebooks/sudoku_dd_verdict_2026-10-28/slides.pdf (python -m notebooks.make_dd_verdict_deck)")

## Caveats

1. **The chained arm's one weak direction.** Editing an emitted deduction makes the prefix inconsistent with the graph and hints, so a model that *detects* the edit and keeps its own answer would score as "does not chain". `reverts_to_original` separates those cases, and for the leak-free arms it is 0.008–0.020 — the misses are that benign case, not a failure to chain. A *high* flip rate carries no such ambiguity, which is why this direction of the result is the safe one.
2. **Forced-node cases only.** Both tests select nodes whose forcing set covers exactly `q-1` colours with a unique binding neighbour. That is DD's elimination rule by construction; it says nothing about instances where DD needs more than one round of bookkeeping to expose such a node.
3. **One seed, one checkpoint per arm**, and `n_colors = 4` so chance is 0.25.
4. **Round depth is capped by the data**: only 4 cases reach round 4, so "no decay" is well powered at rounds 1–2 and suggestive at 3–4.